# 🔥 Project 5.4 — Welding Task Scheduler
**DSA for Mechatronics · Week 05 — Stacks & Queues**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A robotic welding cell receives jobs with different priorities and temperature requirements.
Three data structures work together:

1. **Priority queue** — always starts the highest-priority job next
2. **FIFO queue** — within the same priority, jobs are served in arrival order
3. **Monotonic stack** — for each weld pass, find the *next hotter pass* (useful for heat-affected zone analysis)

Your dataset is a unique set of welding jobs seeded from your student ID.


## Step 1 — Generate your welding job batch

In [ ]:
import heapq
from collections import deque

# ── Personalised parameters ─────────────────────────────────────────
N_JOBS        = random.randint(10, 18)
PRIORITIES    = [1, 2, 3]   # 1 = high, 3 = low
TEMP_RANGE    = (random.randint(600, 800), random.randint(1000, 1400))
DURATION_RANGE= (random.randint(10, 20), random.randint(40, 80))
COOLDOWN      = random.randint(5, 15)   # seconds between jobs

JOB_TYPES  = ["BUTT_WELD","FILLET","SPOT","SEAM","PLUG","TACK","GROOVE"]
jobs = []
for i in range(N_JOBS):
    jobs.append({
        "id":       i + 1,
        "type":     random.choice(JOB_TYPES),
        "priority": random.choice(PRIORITIES),
        "temp_c":   random.randint(*TEMP_RANGE),
        "duration": random.randint(*DURATION_RANGE),
        "arrival":  i * random.randint(1, 4),   # arrival time (seconds)
    })

print(f"Welding job batch parameters:")
print(f"  Jobs           : {N_JOBS}")
print(f"  Temp range     : {TEMP_RANGE[0]} – {TEMP_RANGE[1]} °C")
print(f"  Duration range : {DURATION_RANGE[0]} – {DURATION_RANGE[1]} s")
print(f"  Cooldown       : {COOLDOWN} s between jobs")
print()
print(f"  {'ID':>3}  {'Type':<12}  {'Pri':>4}  {'Temp (°C)':>10}  {'Dur (s)':>8}  {'Arrival':>8}")
print(f"  {'─'*3}  {'─'*12}  {'─'*4}  {'─'*10}  {'─'*8}  {'─'*8}")
for j in jobs:
    print(f"  {j['id']:3d}  {j['type']:<12}  {j['priority']:4d}  {j['temp_c']:10d}  {j['duration']:8d}  {j['arrival']:8d}")


## Step 2 — Priority queue scheduler

In [ ]:
def schedule_jobs(jobs, cooldown):
    """
    Schedule jobs using a priority queue (min-heap on priority).
    Jobs with the same priority are served FIFO (tie-break by arrival time, then id).
    Returns schedule list with start/end times.
    """
    # heap entries: (priority, arrival, id, job_dict)
    heap = []
    for j in jobs:
        heapq.heappush(heap, (j["priority"], j["arrival"], j["id"], j))

    schedule = []
    clock    = 0

    while heap:
        pri, arr, jid, job = heapq.heappop(heap)
        start = max(clock, arr) + (cooldown if schedule else 0)
        end   = start + job["duration"]
        schedule.append({**job, "start": start, "end": end,
                          "wait": start - arr - (cooldown if schedule else 0)})
        clock = end

    return schedule

schedule = schedule_jobs(jobs, COOLDOWN)
total_time   = schedule[-1]["end"]
avg_wait     = round(sum(s["wait"] for s in schedule) / len(schedule), 1)
max_wait     = max(s["wait"] for s in schedule)
max_wait_job = next(s for s in schedule if s["wait"] == max_wait)

print(f"Priority-scheduled execution order:")
print(f"  {'ID':>3}  {'Type':<12}  {'Pri':>4}  {'Start':>7}  {'End':>7}  {'Wait':>6}  {'Temp':>6}")
print(f"  {'─'*3}  {'─'*12}  {'─'*4}  {'─'*7}  {'─'*7}  {'─'*6}  {'─'*6}")
for s in schedule:
    print(f"  {s['id']:3d}  {s['type']:<12}  {s['priority']:4d}  {s['start']:7d}  {s['end']:7d}  {s['wait']:6d}  {s['temp_c']:6d}")

print(f"\n  Total makespan : {total_time} s")
print(f"  Average wait   : {avg_wait} s")
print(f"  Max wait       : {max_wait} s  (job {max_wait_job['id']})")


## Step 3 — FIFO queue: same-priority order vs. priority order comparison

In [ ]:
def schedule_fifo(jobs, cooldown):
    """FIFO scheduler — arrival order only, no priority."""
    fifo = deque(sorted(jobs, key=lambda j: (j["arrival"], j["id"])))
    schedule_f = []
    clock = 0
    while fifo:
        job   = fifo.popleft()
        start = max(clock, job["arrival"]) + (cooldown if schedule_f else 0)
        end   = start + job["duration"]
        schedule_f.append({**job, "start": start, "end": end,
                            "wait": start - job["arrival"] - (cooldown if schedule_f else 0)})
        clock = end
    return schedule_f

fifo_sched   = schedule_fifo(jobs, COOLDOWN)
fifo_total   = fifo_sched[-1]["end"]
fifo_avg_wait = round(sum(s["wait"] for s in fifo_sched) / len(fifo_sched), 1)

# Compare high-priority jobs
hi_pri_jobs = [j["id"] for j in jobs if j["priority"] == 1]
pq_positions  = {s["id"]: i+1 for i, s in enumerate(schedule)}
fifo_positions= {s["id"]: i+1 for i, s in enumerate(fifo_sched)}

print(f"FIFO vs Priority Queue comparison:")
print(f"  {'Metric':<30}  {'FIFO':>10}  {'Priority Q':>12}")
print(f"  {'─'*30}  {'─'*10}  {'─'*12}")
print(f"  {'Total makespan (s)':<30}  {fifo_total:>10}  {total_time:>12}")
print(f"  {'Avg wait time (s)':<30}  {fifo_avg_wait:>10}  {avg_wait:>12}")
print(f"  {'Max wait time (s)':<30}  {max(s["wait"] for s in fifo_sched):>10}  {max_wait:>12}")
print()
if hi_pri_jobs:
    print(f"  High-priority job positions (execution order):")
    print(f"  {'Job ID':>8}  {'FIFO pos':>10}  {'PQ pos':>8}  {'Gain':>6}")
    print(f"  {'─'*8}  {'─'*10}  {'─'*8}  {'─'*6}")
    for jid in hi_pri_jobs:
        fp = fifo_positions.get(jid, "?")
        pp = pq_positions.get(jid, "?")
        gain = fp - pp if isinstance(fp, int) and isinstance(pp, int) else "?"
        print(f"  {jid:8d}  {fp:>10}  {pp:>8}  {gain:>+6}")


## Step 4 — Monotonic stack: next hotter weld pass

In [ ]:
def next_hotter(temps):
    """
    For each weld pass temperature, find the index of the next pass
    that runs hotter. Uses a monotonic stack (decreasing).

    Stack stores indices of passes whose 'next hotter' hasn't been found yet.
    When we find a hotter pass i, we pop all stack entries j where temps[j] < temps[i].

    O(n) time — each index pushed/popped at most once.
    Returns list of (index, temp, next_hotter_index_or_None).
    """
    n      = len(temps)
    result = [-1] * n      # -1 = no hotter pass found
    stack  = []            # indices, monotonically decreasing temps

    for i in range(n):
        while stack and temps[stack[-1]] < temps[i]:
            j = stack.pop()
            result[j] = i   # i is the next hotter pass for j
        stack.append(i)

    return result

exec_temps = [s["temp_c"] for s in schedule]
next_hot   = next_hotter(exec_temps)

print(f"Monotonic stack — next hotter weld pass:")
print(f"  {'Exec#':>6}  {'Job ID':>7}  {'Temp (°C)':>10}  {'Next hotter at exec#':>22}")
print(f"  {'─'*6}  {'─'*7}  {'─'*10}  {'─'*22}")
for i, (s, nh) in enumerate(zip(schedule, next_hot)):
    nh_str = f"exec #{nh+1}  ({exec_temps[nh]} °C)" if nh != -1 else "none"
    print(f"  {i+1:6d}  {s['id']:7d}  {s['temp_c']:10d}  {nh_str:>22}")

no_hotter = sum(1 for x in next_hot if x == -1)
max_gap   = max((nh - i for i, nh in enumerate(next_hot) if nh != -1), default=0)
print(f"\n  Passes with no hotter successor: {no_hotter}")
print(f"  Max gap to next hotter pass    : {max_gap} positions")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " WELDING TASK SCHEDULER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  JOB BATCH" + " "*(W-11) + "║")
print(f"║  {'Jobs':<28}: {N_JOBS:<26}║")
print(f"║  {'Temperature range':<28}: {TEMP_RANGE[0]}–{TEMP_RANGE[1]} °C{'':<18}║")
print(f"║  {'Cooldown between jobs':<28}: {COOLDOWN} s{'':<24}║")
print("╠" + "═"*W + "╣")
print("║  SCHEDULER COMPARISON" + " "*(W-22) + "║")
print(f"║  {'FIFO makespan':<28}: {fifo_total} s{'':<24}║")
print(f"║  {'Priority Q makespan':<28}: {total_time} s{'':<24}║")
print(f"║  {'FIFO avg wait':<28}: {fifo_avg_wait} s{'':<24}║")
print(f"║  {'Priority Q avg wait':<28}: {avg_wait} s{'':<24}║")
print(f"║  {'Max wait job':<28}: Job {max_wait_job["id"]} ({max_wait} s){'':<14}║")
print("╠" + "═"*W + "╣")
print("║  HEAT ANALYSIS (MONOTONIC STACK)" + " "*(W-33) + "║")
print(f"║  {'Passes with no hotter next':<28}: {no_hotter:<26}║")
print(f"║  {'Max gap to next hotter':<28}: {max_gap} positions{'':<18}║")
print(f"║  {'Hottest pass temp':<28}: {max(exec_temps)} °C{'':<22}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which stack or queue concept did you find most important, and why?**
Pick one technique from the notebook and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
